# Module 27 — Word2Vec with Gensim: Query Semantics

> **Track 3 · Yandex Big Tech & Search**  
> **Difficulty**: Intermediate  
> **Runtime**: ~5 minutes  
> **Key Libraries**: Gensim, Pandas, NumPy, Scikit-learn, Plotly  
> **Prerequisite**: Module 25 (Text Preprocessing)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Search Query Logs](#3-synthetic-search-query-logs)
4. [Word2Vec Training](#4-word2vec-training)
5. [Query Expansion](#5-query-expansion)
6. [Semantic Similarity](#6-semantic-similarity)
7. [Visualization](#7-visualization)
8. [Key Business Takeaway](#8-key-business-takeaway)

---
## §1 · Business Context

### Why Word2Vec for search?

Word2Vec learns **semantic embeddings** from text:
- **Dense representations**: Capture word meaning in vectors.
- **Semantic similarity**: Find related words automatically.
- **Query expansion**: Improve search recall with synonyms.

**Applications at Yandex**:
1. **Query understanding**: Expand queries with related terms.
2. **Semantic search**: Find documents with similar meaning.
3. **Recommendation**: Suggest related products/content.

**Key concepts**:
- **CBOW**: Predict word from context.
- **Skip-gram**: Predict context from word.
- **Embedding space**: Words with similar meanings are close.

In this notebook we will:
1. Generate synthetic search query logs.
2. Train Word2Vec models (CBOW and Skip-gram).
3. Perform query expansion and semantic similarity.
4. Visualize word embeddings with t-SNE.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── NLP libraries ────────────────────────────────────────────────
from gensim.models import Word2Vec
from sklearn.manifold import TSNE

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 100)
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")

---
## §3 · Synthetic Search Query Logs

In [ ]:
# Generate synthetic search query logs
N_QUERIES = 50000

# Query templates with semantic relationships
query_templates = {
    'electronics': [
        'best {product} 2024',
        'cheap {product} deals',
        '{product} reviews',
        'buy {product} online',
        '{product} vs {product2}',
    ],
    'clothing': [
        '{brand} {product}',
        '{product} for {season}',
        '{color} {product}',
        '{product} sale',
    ],
    'food': [
        '{cuisine} restaurant near me',
        'best {food} in {city}',
        '{food} recipe',
        'healthy {food}',
    ]
}

# Word lists
electronics_products = ['laptop', 'phone', 'headphones', 'camera', 'tablet', 'smartwatch']
clothing_products = ['dress', 'shoes', 'jacket', 'jeans', 't-shirt']
brands = ['nike', 'adidas', 'zara', 'h&m', 'gucci']
cuisines = ['italian', 'chinese', 'japanese', 'mexican', 'indian']
foods = ['pizza', 'sushi', 'burger', 'pasta', 'salad']
cities = ['moscow', 'new york', 'london', 'tokyo', 'paris']

# Generate queries
queries = []
for _ in range(N_QUERIES):
    category = np.random.choice(list(query_templates.keys()))
    template = np.random.choice(query_templates[category])
    
    if category == 'electronics':
        query = template.format(
            product=np.random.choice(electronics_products),
            product2=np.random.choice(electronics_products)
        )
    elif category == 'clothing':
        query = template.format(
            brand=np.random.choice(brands),
            product=np.random.choice(clothing_products),
            season=np.random.choice(['summer', 'winter', 'spring', 'fall']),
            color=np.random.choice(['red', 'blue', 'black', 'white', 'green'])
        )
    else:
        query = template.format(
            cuisine=np.random.choice(cuisines),
            food=np.random.choice(foods),
            city=np.random.choice(cities)
        )
    
    queries.append(query.lower().split())

print(f"✓ Generated {N_QUERIES} search queries")
print(f"\nSample queries:")
for i in range(5):
    print(f"  {' '.join(queries[i])}")

---
## §4 · Word2Vec Training

In [ ]:
# Train Word2Vec models
# CBOW model
print("Training CBOW model...")
model_cbow = Word2Vec(
    sentences=queries,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=0,  # CBOW
    epochs=10
)

# Skip-gram model
print("Training Skip-gram model...")
model_sg = Word2Vec(
    sentences=queries,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,  # Skip-gram
    epochs=10
)

print(f"✓ Models trained")
print(f"  Vocabulary size: {len(model_cbow.wv)}")
print(f"  Embedding dimension: {model_cbow.vector_size}")

---
## §5 · Query Expansion

In [ ]:
# Query expansion using Word2Vec
def expand_query(query, model, topn=5):
    """Expand query with similar words."""
    tokens = query.lower().split()
    expanded = set(tokens)
    
    for token in tokens:
        if token in model.wv:
            similar_words = model.wv.most_similar(token, topn=topn)
            for word, score in similar_words:
                if score > 0.6:  # Similarity threshold
                    expanded.add(word)
    
    return list(expanded)

# Test query expansion
test_queries = [
    'best laptop 2024',
    'cheap headphones deals',
    'nike shoes sale'
]

print("=" * 70)
print("QUERY EXPANSION EXAMPLES")
print("=" * 70)

for query in test_queries:
    expanded = expand_query(query, model_cbow)
    print(f"\nOriginal: {query}")
    print(f"Expanded: {' '.join(expanded[:10])}...")

---
## §6 · Semantic Similarity

In [ ]:
# Calculate semantic similarity between words
def get_similarity(word1, word2, model):
    """Get similarity between two words."""
    if word1 in model.wv and word2 in model.wv:
        return model.wv.similarity(word1, word2)
    return None

# Test similarities
word_pairs = [
    ('laptop', 'computer'),
    ('laptop', 'phone'),
    ('laptop', 'shoes'),
    ('nike', 'adidas'),
    ('nike', 'pizza'),
]

print("=" * 70)
print("SEMANTIC SIMILARITY EXAMPLES")
print("=" * 70)

for word1, word2 in word_pairs:
    sim = get_similarity(word1, word2, model_cbow)
    if sim:
        print(f"\n{word1} ↔ {word2}: {sim:.4f}")
    else:
        print(f"\n{word1} ↔ {word2}: Word not in vocabulary")

---
## §7 · Visualization

In [ ]:
# Visualize word embeddings with t-SNE
# Get embeddings for common words
words_to_plot = ['laptop', 'phone', 'headphones', 'camera', 'tablet',
                 'nike', 'adidas', 'shoes', 'dress', 'jacket',
                 'pizza', 'sushi', 'burger', 'italian', 'chinese']

# Filter words in vocabulary
words_in_vocab = [w for w in words_to_plot if w in model_cbow.wv]
embeddings = np.array([model_cbow.wv[w] for w in words_in_vocab])

# Apply t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=5)
embeddings_2d = tsne.fit_transform(embeddings)

# Create visualization
fig = px.scatter(
    x=embeddings_2d[:, 0],
    y=embeddings_2d[:, 1],
    text=words_in_vocab,
    title='Word2Vec Embeddings (t-SNE)',
    labels={'x': 't-SNE 1', 'y': 't-SNE 2'}
)

fig.update_traces(textposition='top center', marker=dict(size=10))
fig.update_layout(height=500)
fig.show()

print("💡 Key insights:")
print("   - Similar words cluster together")
print("   - Electronics products form one cluster")
print("   - Clothing items form another cluster")
print("   - Food-related words cluster separately")

---
## §8 · Key Business Takeaway

> ### 🎯 Action Items for Search & Recommendation Teams
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | Query expansion, semantic search, recommendations |
> | **Model** | Skip-gram for rare words, CBOW for frequent words |
> | **Training** | Use domain-specific corpus (search logs) |
> | **Evaluation** | Word similarity tasks, query expansion quality |
> | **Deployment** | Pre-compute embeddings, cache similar words |
>
> ### 💡 Yandex-Specific Guidelines
>
> 1. **Query expansion pipeline**:
>    ```
>    Query → Tokenize → Expand with Word2Vec → Search expanded query
>    ```
>
> 2. **Use cases**:
>    | Use Case | Benefit |
>    |----------|--------|
>    | Query expansion | Improve recall by 10-20% |
>    | Semantic matching | Find relevant docs with different words |
>    | Recommendations | "Users who searched X also searched Y" |
>
> 3. **Production tips**:
>    - Retrain embeddings monthly on fresh query logs
>    - Use approximate nearest neighbors for fast lookup
>    - A/B test expansion to measure impact
>
> ### 🔑 Key Takeaways
>
> 1. **Word2Vec captures semantic relationships** between words.
> 2. **Query expansion improves recall** by including synonyms.
> 3. **Skip-gram works better** for rare words and small corpora.
> 4. **Domain-specific training** is essential for good embeddings.
> 5. **Visualize embeddings** to verify semantic clustering.